# Projet: Analyse des offres d'emploi LinkedIn avec Snowflake


## 1. Création Database


In [ ]:
%%sql -r dataframe_1
Create database if not exists LINKEDIN;

## 2. Création d'un stage externe


In [ ]:
%%sql -r dataframe_2
USE DATABASE LINKEDIN;
Create stage if not exists LINKEDIN_DATA
url = 's3://snowflake-lab-bucket/' ;

In [ ]:
%%sql -r dataframe_5
List @Linkedin_data

## Création format CSV & JSON

In [ ]:
%%sql -r dataframe_3
  field_delimiter = ','
  record_delimiter = '\n'
  skip_header = 1
  field_optionally_enclosed_by = '\042'
  null_if = (''); 

CREATE OR REPLACE FILE FORMAT JSON_FORMAT
  type = 'JSON'
  strip_outer_array = true;

## Création des tables json

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE TABLE  json_companies_data (
 v variant
);

CREATE OR REPLACE TABLE json_company_industries_data (
 v variant
);

CREATE OR REPLACE TABLE json_company_specialities_data (
 v variant
);

COPY INTO json_companies_data
FROM @LINKEDIN_DATA/companies.json
FILE_FORMAT = JSON_FORMAT;

COPY INTO json_company_industries_data
FROM @LINKEDIN_DATA/company_industries.json
FILE_FORMAT = JSON_FORMAT;

COPY INTO json_company_specialities_data
FROM @LINKEDIN_DATA/company_specialities.json
FILE_FORMAT = JSON_FORMAT;

CREATE OR REPLACE TABLE Companies as
select 
    v:"company_id"::STRING as "company_id" ,
    v:"name"::STRING as "name",
    v:"description"::STRING as "description",
    v:"company_size"::INTEGER as "company_size",
    v:"state"::STRING as "state",
    v:"country"::STRING as "country",
    v:"city"::STRING as "city",
    v:"zip_code"::STRING as "zip_code",
    v:"address"::TIMESTAMP as "address",
    v:"url"::STRING as "url" 
from json_companies_data;


CREATE OR REPLACE TABLE Company_industries as
select 
    v:"company_id"::STRING as "company_id" ,
    v:"industry"::STRING as "industry"
from json_company_industries_data;


CREATE OR REPLACE TABLE Company_specialities as
select 
    v:"company_id"::STRING as "company_id" ,
    v:"speciality"::STRING as "speciality"
from json_company_specialities_data;

In [ ]:
%%sql -r dataframe_7
select * from json_companies_data;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE JOB_POSTINGS (
    job_id                      STRING          NOT NULL,
    company_name                VARCHAR(255),
    title                       VARCHAR(255),
    description                 TEXT,
    max_salary                  FLOAT,
    med_salary                  FLOAT,
    min_salary                  FLOAT,
    pay_period                  VARCHAR(50),        -- Hourly, Monthly, Yearly
    formatted_work_type         VARCHAR(50),        -- Fulltime, Parttime, Contract
    location                    VARCHAR(255),
    applies                     STRING,
    original_listed_time        TIMESTAMP_NTZ,
    remote_allowed              BOOLEAN,
    views                       STRING,
    job_posting_url             VARCHAR(1000),
    application_url             VARCHAR(1000),
    application_type            VARCHAR(100),       -- offsite, complex/simple onsite
    expiry                      TIMESTAMP_NTZ,
    closed_time                 TIMESTAMP_NTZ,
    formatted_experience_level  VARCHAR(100),       -- entry, associate, executive…
    skills_desc                 TEXT,
    listed_time                 TIMESTAMP_NTZ,
    posting_domain              VARCHAR(255),
    sponsored                   BOOLEAN,
    work_type                   VARCHAR(100),
    currency                    VARCHAR(10),
    compensation_type           VARCHAR(100),
 
    CONSTRAINT pk_job_postings PRIMARY KEY (job_id)
);

CREATE OR REPLACE TABLE JOB_SKILLS (
    job_id      STRING          NOT NULL,
    skill_abr   VARCHAR(50)     NOT NULL,   -- abréviation de la compétence
 
    CONSTRAINT pk_job_skills PRIMARY KEY (job_id, skill_abr),
    CONSTRAINT fk_job_skills_job FOREIGN KEY (job_id) REFERENCES JOB_POSTINGS(job_id)
);

CREATE OR REPLACE TABLE JOB_INDUSTRIES (
    job_id          STRING  NOT NULL,
    industry_id     STRING  NOT NULL,
 
    CONSTRAINT pk_job_industries PRIMARY KEY (job_id, industry_id),
    CONSTRAINT fk_job_industries_job FOREIGN KEY (job_id) REFERENCES JOB_POSTINGS(job_id)
);

COPY INTO JOB_POSTINGS
FROM @LINKEDIN_DATA/job_postings.csv
FILE_FORMAT = CSV;

COPY INTO JOB_SKILLS
FROM @LINKEDIN_DATA/job_skills.csv
FILE_FORMAT = CSV;

COPY INTO JOB_INDUSTRIES
FROM @LINKEDIN_DATA/job_industries.csv
FILE_FORMAT = CSV;


# 